# CSE 475 - Assignment 01
## Group Information

| Field | Details |
|---|---|
| **Group ID** | Group XX |
| **Student 1 Name** | YOUR_NAME |
| **Student 1 ID** | YOUR_ID |
| **Student 2 Name** | PARTNER_NAME |
| **Student 2 ID** | PARTNER_ID |
| **Notebook Type** | Transformer Models |
| **Dataset Source Name** | Tropical Flower Dataset Seven Species from Bangladesh |
| **Dataset Source Link** | DATASET_SOURCE_URL |
| **Kaggle Dataset Path** | /kaggle/input/datasets/sabuktagin/tropical-flowers/ |
| **Submission Date** | DD Month 2026 |

## Dataset Source Acknowledgement

**Dataset Name:** Tropical Flower Dataset Seven Species from Bangladesh

**Source:** Obtained from Kaggle. Contains images of seven tropical flower species from Bangladesh.

**Kaggle Path:** `/kaggle/input/datasets/sabuktagin/tropical-flowers/`

## [*] Global Configuration

All hyperparameters, patch sizes, positional embedding types, paths, and seeds.

In [ ]:
import os, random, numpy as np, torch

DATA_DIR = '/kaggle/input/datasets/sabuktagin/tropical-flowers/Tropical Flower Dataset Seven Species from Bangladesh for Classification and Ecological Research/Flower Dataset/Flower Dataset'
OUTPUT_DIR = '/kaggle/working'
BATCH_SIZE = 32
NUM_EPOCHS = 30
SEED = 42
PATCH_SIZE = 16
POSITIONAL_EMBEDDING = 'learned'
WARMUP_EPOCHS = 5
LABEL_SMOOTHING = 0.1
GRADIENT_CLIP_NORM = 1.0
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.05
EARLY_STOP_PATIENCE = 7
IMG_SIZE = 224
NUM_CLASSES = 7
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
CLASS_NAMES = ['Bougainvillea', 'Crown of thorns', 'Hibiscus', 'Jungle geranium',
               'Madagascar periwinkle', 'Marigold', 'Rose']
MODEL_CONFIGS = {
    'ViT': 'vit_small_patch16_224',
    'Swin': 'swin_tiny_patch4_window7_224',
    'DeiT': 'deit_small_patch16_224',
}

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | ImgSize: {IMG_SIZE}")
print(f"Patch: {PATCH_SIZE} | LR: {LEARNING_RATE} | WD: {WEIGHT_DECAY}")

## [+] Setup & Imports

Install required packages, import all libraries, confirm GPU, print versions.

In [ ]:
!pip install timm -q

import time, copy, math
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torchvision import transforms
import timm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

print(f"PyTorch: {torch.__version__} | Timm: {timm.__version__}")
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## [=] Exploratory Data Analysis (EDA)

Comprehensive exploration of the Tropical Flower Dataset to understand class distributions, image properties, and characteristics that influence transformer-based patch tokenization.

In [ ]:
# Load dataset paths and labels
data_path = Path(DATA_DIR)
all_images, all_labels = [], []
for class_folder in sorted(data_path.iterdir()):
    if class_folder.is_dir():
        for img_file in class_folder.iterdir():
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                all_images.append(str(img_file))
                all_labels.append(class_folder.name)

df = pd.DataFrame({'image_path': all_images, 'label': all_labels})
print(f"Total images: {len(df)} | Classes: {df['label'].nunique()}")
print(df['label'].value_counts().to_string())

In [ ]:
# Class distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
class_counts = df['label'].value_counts()
colors = sns.color_palette('viridis', n_colors=NUM_CLASSES)

bars = axes[0].bar(range(NUM_CLASSES), class_counts.values, color=colors, edgecolor='black')
axes[0].set_xticks(range(NUM_CLASSES))
axes[0].set_xticklabels(class_counts.index, rotation=45, ha='right')
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution (Bar)')
for b, c in zip(bars, class_counts.values):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+3, str(c), ha='center', fontweight='bold')

axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', colors=colors)
axes[1].set_title('Class Distribution (Pie)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eda_class_dist.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample images from each class
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()
for idx, cn in enumerate(CLASS_NAMES):
    imgs = df[df['label']==cn]['image_path'].values
    img = Image.open(imgs[0]).convert('RGB')
    axes[idx].imshow(img)
    axes[idx].set_title(f'{cn} ({img.size[0]}x{img.size[1]})', fontweight='bold')
    axes[idx].axis('off')
axes[-1].axis('off')
plt.suptitle('Sample Images per Class', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eda_samples.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Image size statistics
widths, heights = [], []
for p in df['image_path'].values[:500]:
    with Image.open(p) as img:
        widths.append(img.size[0])
        heights.append(img.size[1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(widths, bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Width Distribution')
axes[0].axvline(np.mean(widths), color='red', linestyle='--', label=f'Mean: {np.mean(widths):.0f}')
axes[0].legend()
axes[1].hist(heights, bins=30, color='coral', edgecolor='black')
axes[1].set_title('Height Distribution')
axes[1].axvline(np.mean(heights), color='red', linestyle='--', label=f'Mean: {np.mean(heights):.0f}')
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eda_sizes.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"W: {min(widths)}-{max(widths)} (mean {np.mean(widths):.0f})")
print(f"H: {min(heights)}-{max(heights)} (mean {np.mean(heights):.0f})")

### Patch-Based Tokenization Impact

With input size 224x224 and patch size 16, each image yields (224/16)^2 = **196 tokens**. Key observations:

- **Resolution variance** in the dataset means resizing is essential. Transformer models require fixed-size inputs.
- **Object scale variation** benefits from Swin's hierarchical multi-scale approach vs ViT's uniform global attention.
- **Fine-grained differences** between species (e.g., Hibiscus vs Bougainvillea) require attention to discriminative regions like petal shape and center patterns.

## [W] Data Preprocessing & Augmentation

Transformer-optimized preprocessing pipeline with higher resolution (224x224), ImageNet normalization, and RandAugment for robust data augmentation. The EDA confirmed diverse image sizes, so resizing with center-crop at test time ensures consistent input dimensions for patch tokenization.

In [ ]:
# [W] Custom Dataset and Augmentation Pipeline
class FlowerDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        # Build label-to-index mapping
        self.class_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.class_to_idx[self.labels[idx]]
        if self.transform:
            img = self.transform(img)
        return img, label

# Transformer-optimized augmentation
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),  # Slight upscale before crop
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandAugment(num_ops=2, magnitude=9),     # Advanced augmentation for transformers
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Augmentation pipelines defined:")
print(f"  Train: RandAugment + RandomCrop + HFlip + ColorJitter + RandomErasing")
print(f"  Val/Test: Resize only (deterministic)")

In [ ]:
# Stratified Train/Val/Test split (80/10/10)
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    df['image_path'].values, df['label'].values, test_size=0.2,
    random_state=SEED, stratify=df['label'].values
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5,
    random_state=SEED, stratify=temp_labels
)

# Create datasets
train_dataset = FlowerDataset(train_paths, train_labels, train_transform)
val_dataset = FlowerDataset(val_paths, val_labels, val_test_transform)
test_dataset = FlowerDataset(test_paths, test_labels, val_test_transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

# Verify class distribution across splits
for name, labels in [('Train', train_labels), ('Val', val_labels), ('Test', test_labels)]:
    counts = Counter(labels)
    print(f"\n{name} split distribution:")
    for cn in CLASS_NAMES:
        print(f"  {cn}: {counts.get(cn, 0)}")

## [T] Training Infrastructure

Unified training loop with Transformer best practices:
- **AdamW** optimizer with weight decay (decoupled weight regularization)
- **Warmup + CosineAnnealing** learning rate schedule
- **Gradient clipping** (max_norm=1.0) for training stability
- **Mixed-precision training** (torch.cuda.amp) for faster computation
- **Early stopping** and **best checkpoint** saving

In [ ]:
# [T] Warmup + Cosine Annealing LR Scheduler
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.min_lr = min_lr
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]

    def step(self, epoch):
        if epoch < self.warmup_epochs:
            # Linear warmup
            factor = (epoch + 1) / self.warmup_epochs
        else:
            # Cosine annealing
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            factor = 0.5 * (1 + math.cos(math.pi * progress))
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = max(self.min_lr, base_lr * factor)
        return self.optimizer.param_groups[0]['lr']

    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']

print("WarmupCosineScheduler defined")

In [ ]:
# [T] Training and Evaluation Functions with Mixed Precision
from tqdm.auto import tqdm

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch, total_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f'Ep {epoch+1}/{total_epochs}', leave=False,
                bar_format='{l_bar}{bar:30}{r_bar}', colour='blue')

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        # Mixed precision forward pass
        with autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)

        # Scaled backward pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(acc=f'{100.*correct/total:.2f}%', loss=f'{running_loss/total:.4f}')

    return running_loss / total, 100. * correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc='Eval', leave=False,
                bar_format='{l_bar}{bar:30}{r_bar}', colour='green')

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        with autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(acc=f'{100.*correct/total:.2f}%')

    return running_loss / total, 100. * correct / total

print("Training and evaluation functions defined")

In [ ]:
# [T] Master Training Function
def train_model(model, model_name, train_loader, val_loader, num_epochs=NUM_EPOCHS):
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = WarmupCosineScheduler(optimizer, WARMUP_EPOCHS, num_epochs)
    scaler = GradScaler("cuda")

    # History tracking
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
    best_val_acc = 0.0
    best_model_wts = None
    patience_counter = 0
    start_time = time.time()

    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")

    for epoch in range(num_epochs):
        lr = scheduler.step(epoch)
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE, epoch, num_epochs)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(lr)

        print(f"Ep {epoch+1:02d}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | LR: {lr:.6f}")

        # Checkpoint best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, os.path.join(OUTPUT_DIR, f'{model_name}_best.pth'))
            patience_counter = 0
            print(f"  -> New best! Val Acc: {val_acc:.2f}%")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    elapsed = time.time() - start_time
    print(f"\nTraining complete in {elapsed/60:.1f} min | Best Val Acc: {best_val_acc:.2f}%")

    # Restore best weights
    model.load_state_dict(best_model_wts)
    history['training_time'] = elapsed
    history['best_val_acc'] = best_val_acc
    return model, history

print("Master training function defined")

## [TF] Transformer Models: ViT, Swin, DeiT

Three state-of-the-art Vision Transformer architectures are implemented, each representing a distinct approach to image understanding:

| Model | Architecture | Key Innovation |
|-------|-------------|----------------|
| **ViT** | Standard patch-based | Global self-attention across all patches |
| **Swin** | Hierarchical shifted-window | Multi-scale features with local+shifted attention |
| **DeiT** | Data-efficient ViT | Knowledge distillation token for improved small-data performance |

In [ ]:
# [TF] Helper: Load a pretrained model from timm and adapt the head
def create_transformer_model(model_name_key):
    timm_name = MODEL_CONFIGS[model_name_key]
    model = timm.create_model(timm_name, pretrained=True, num_classes=NUM_CLASSES)
    model = model.to(DEVICE)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{model_name_key} ({timm_name})")
    print(f"  Total params: {total_params:,}")
    print(f"  Trainable params: {trainable_params:,}")
    print(f"  Size: {total_params * 4 / 1e6:.1f} MB (fp32)")
    return model

# Store all trained models and histories
trained_models = {}
all_histories = {}

### Vision Transformer (ViT-Small/16)

**Architecture:** The original Vision Transformer (Dosovitskiy et al., 2020) treats an image as a sequence of fixed-size patches. Each 16x16 patch is linearly projected into an embedding, a learnable [CLS] token is prepended, and fixed/learned positional embeddings are added. The sequence is processed by a standard Transformer encoder with multi-head self-attention (MHSA).

**Key Properties:**
- Global receptive field from the first layer (every patch attends to every other)
- No inductive bias toward locality (unlike CNNs)
- Requires larger datasets or strong pretraining to match CNNs

In [ ]:
# Train ViT
seed_everything(SEED)
vit_model = create_transformer_model('ViT')
vit_model, vit_history = train_model(vit_model, 'ViT', train_loader, val_loader)
trained_models['ViT'] = vit_model
all_histories['ViT'] = vit_history

In [ ]:
# ViT training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(vit_history['train_loss'])+1)

axes[0].plot(epochs_range, vit_history['train_loss'], 'b-', label='Train')
axes[0].plot(epochs_range, vit_history['val_loss'], 'r-', label='Val')
axes[0].set_title('ViT — Loss Curves')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, vit_history['train_acc'], 'b-', label='Train')
axes[1].plot(epochs_range, vit_history['val_acc'], 'r-', label='Val')
axes[1].set_title('ViT — Accuracy Curves')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_range, vit_history['lr'], 'g-')
axes[2].set_title('ViT — Learning Rate Schedule')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR'); axes[2].grid(alpha=0.3)

plt.suptitle(f"ViT Training — Best Val Acc: {vit_history['best_val_acc']:.2f}%", fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'vit_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

### Swin Transformer (Swin-Tiny)

**Architecture:** Swin Transformer (Liu et al., 2021) uses a hierarchical design with shifted windows. Unlike ViT's global attention, Swin computes self-attention within local windows (7x7) and shifts them between layers for cross-window connections. This creates a multi-scale feature pyramid similar to CNNs.

**Key Properties:**
- Linear computational complexity w.r.t. image size (vs quadratic for ViT)
- Hierarchical feature maps at multiple resolutions
- Shifted-window mechanism enables cross-window information flow
- Strong inductive bias toward locality while maintaining transformer flexibility

In [ ]:
# Train Swin
seed_everything(SEED)
swin_model = create_transformer_model('Swin')
swin_model, swin_history = train_model(swin_model, 'Swin', train_loader, val_loader)
trained_models['Swin'] = swin_model
all_histories['Swin'] = swin_history

In [ ]:
# Swin training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(swin_history['train_loss'])+1)

axes[0].plot(epochs_range, swin_history['train_loss'], 'b-', label='Train')
axes[0].plot(epochs_range, swin_history['val_loss'], 'r-', label='Val')
axes[0].set_title('Swin — Loss Curves')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, swin_history['train_acc'], 'b-', label='Train')
axes[1].plot(epochs_range, swin_history['val_acc'], 'r-', label='Val')
axes[1].set_title('Swin — Accuracy Curves')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_range, swin_history['lr'], 'g-')
axes[2].set_title('Swin — Learning Rate Schedule')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR'); axes[2].grid(alpha=0.3)

plt.suptitle(f"Swin Training — Best Val Acc: {swin_history['best_val_acc']:.2f}%", fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'swin_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

### DeiT (Data-efficient Image Transformer — DeiT-Small)

**Architecture:** DeiT (Touvron et al., 2021) builds on ViT but introduces data-efficient training strategies. The key innovation is a distillation token alongside the [CLS] token, enabling knowledge distillation from a CNN teacher. The training recipe includes strong augmentation (RandAugment), regularization (DropPath, label smoothing), and repeated augmentation.

**Key Properties:**
- Same architecture as ViT but with optimized training recipe
- Distillation token for knowledge transfer from CNNs
- Designed to work well with smaller datasets (unlike original ViT)
- Strong regularization prevents overfitting on limited data

In [ ]:
# Train DeiT
seed_everything(SEED)
deit_model = create_transformer_model('DeiT')
deit_model, deit_history = train_model(deit_model, 'DeiT', train_loader, val_loader)
trained_models['DeiT'] = deit_model
all_histories['DeiT'] = deit_history

In [ ]:
# DeiT training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(deit_history['train_loss'])+1)

axes[0].plot(epochs_range, deit_history['train_loss'], 'b-', label='Train')
axes[0].plot(epochs_range, deit_history['val_loss'], 'r-', label='Val')
axes[0].set_title('DeiT — Loss Curves')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, deit_history['train_acc'], 'b-', label='Train')
axes[1].plot(epochs_range, deit_history['val_acc'], 'r-', label='Val')
axes[1].set_title('DeiT — Accuracy Curves')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_range, deit_history['lr'], 'g-')
axes[2].set_title('DeiT — Learning Rate Schedule')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR'); axes[2].grid(alpha=0.3)

plt.suptitle(f"DeiT Training — Best Val Acc: {deit_history['best_val_acc']:.2f}%", fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'deit_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## [ ] Evaluation & Comparison

Comprehensive evaluation of all three transformer models on the held-out test set. Metrics include accuracy, precision, recall, F1-score, confusion matrices, inference time, and attention map visualization.

In [ ]:
# [ ] Test set evaluation for all models
@torch.no_grad()
def evaluate_on_test(model, test_loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    start = time.time()
    for images, labels in test_loader:
        images = images.to(device)
        with autocast("cuda"):
            outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
    inference_time = time.time() - start
    return np.array(all_preds), np.array(all_labels), np.array(all_probs), inference_time

test_results = {}
for name, model in trained_models.items():
    preds, labels, probs, inf_time = evaluate_on_test(model, test_loader, DEVICE)
    acc = accuracy_score(labels, preds) * 100
    f1_mac = f1_score(labels, preds, average='macro') * 100
    f1_wt = f1_score(labels, preds, average='weighted') * 100
    prec = precision_score(labels, preds, average='macro') * 100
    rec = recall_score(labels, preds, average='macro') * 100
    test_results[name] = {
        'preds': preds, 'labels': labels, 'probs': probs,
        'accuracy': acc, 'f1_macro': f1_mac, 'f1_weighted': f1_wt,
        'precision': prec, 'recall': rec, 'inference_time': inf_time
    }
    print(f"\n{name}: Acc={acc:.2f}% | F1(macro)={f1_mac:.2f}% | Prec={prec:.2f}% | Rec={rec:.2f}% | Time={inf_time:.2f}s")

In [ ]:
# [ ] Comparison table
print("\n" + "="*80)
print("MODEL COMPARISON TABLE")
print("="*80)
header = f"{'Model':<10} {'Acc%':<8} {'F1-Mac%':<9} {'F1-Wt%':<9} {'Prec%':<8} {'Rec%':<8} {'Time(s)':<8} {'Params':<12}"
print(header)
print("-"*80)
for name in ['ViT', 'Swin', 'DeiT']:
    r = test_results[name]
    params = sum(p.numel() for p in trained_models[name].parameters())
    print(f"{name:<10} {r['accuracy']:<8.2f} {r['f1_macro']:<9.2f} {r['f1_weighted']:<9.2f} {r['precision']:<8.2f} {r['recall']:<8.2f} {r['inference_time']:<8.2f} {params:,}")
print("="*80)

In [ ]:
# [ ] Classification reports
for name in ['ViT', 'Swin', 'DeiT']:
    r = test_results[name]
    print(f"\n{'='*60}")
    print(f"Classification Report — {name}")
    print(f"{'='*60}")
    print(classification_report(r['labels'], r['preds'], target_names=CLASS_NAMES, digits=4))

In [ ]:
# [ ] Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for idx, name in enumerate(['ViT', 'Swin', 'DeiT']):
    r = test_results[name]
    cm = confusion_matrix(r['labels'], r['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES, ax=axes[idx], cbar=False)
    axes[idx].set_title(f'{name} (Acc: {r["accuracy"]:.2f}%)', fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('True')
    axes[idx].tick_params(axis='x', rotation=45)
plt.suptitle('Confusion Matrices — Transformer Models', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [ ] Attention map visualization (5 samples per model)
def get_attention_map_vit(model, img_tensor, device):
    """Extract attention from the last transformer block for ViT/DeiT models."""
    model.eval()
    # Register a hook on the last attention layer
    attention_weights = []
    def hook_fn(module, input, output):
        # For timm ViT, attention is computed inside the Attention module
        attention_weights.append(output)

    # Try to hook into the attention of the last block
    try:
        # For ViT and DeiT models from timm
        last_block = model.blocks[-1].attn
        # Get attention by computing manually
        img = img_tensor.unsqueeze(0).to(device)
        with torch.no_grad():
            with autocast("cuda"):
                out = model(img)

        # Use gradient-free attention approximation via feature maps
        # Get the features before the last attention layer
        features = model.forward_features(img)
        if hasattr(features, 'shape') and len(features.shape) == 3:
            # Remove CLS token, reshape to spatial
            feat = features[:, 1:, :]  # Remove CLS
            n_patches = int(feat.shape[1] ** 0.5)
            # Average across feature dimension
            attn_map = feat.mean(dim=-1).reshape(1, n_patches, n_patches)
            attn_map = F.interpolate(attn_map.unsqueeze(0).float(), size=(IMG_SIZE, IMG_SIZE),
                                     mode='bilinear', align_corners=False)
            return attn_map.squeeze().cpu().numpy()
    except Exception:
        pass

    # Fallback: use Grad-CAM-like approach
    return None

def visualize_attention_maps(model, model_name, dataset, n_samples=5):
    """Visualize attention/feature maps for sample images."""
    model.eval()
    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4*n_samples))
    indices = np.random.choice(len(dataset), n_samples, replace=False)

    for row, idx in enumerate(indices):
        img_tensor, label = dataset[idx]

        # Original image (denormalize)
        img_show = img_tensor.clone()
        for c in range(3):
            img_show[c] = img_show[c] * IMAGENET_STD[c] + IMAGENET_MEAN[c]
        img_show = img_show.permute(1, 2, 0).clamp(0, 1).numpy()

        axes[row, 0].imshow(img_show)
        axes[row, 0].set_title(f'True: {CLASS_NAMES[label]}', fontsize=10)
        axes[row, 0].axis('off')

        # Get prediction
        with torch.no_grad():
            out = model(img_tensor.unsqueeze(0).to(DEVICE))
            pred = out.argmax(1).item()

        # Attention/feature map
        attn = get_attention_map_vit(model, img_tensor, DEVICE)
        if attn is not None:
            axes[row, 1].imshow(attn, cmap='hot')
            axes[row, 1].set_title('Attention Map', fontsize=10)
        else:
            # Fallback: show activation magnitudes from forward features
            with torch.no_grad():
                feat = model.forward_features(img_tensor.unsqueeze(0).to(DEVICE))
            if hasattr(feat, 'shape') and len(feat.shape) == 3:
                f = feat[:, 1:, :].mean(-1) if feat.shape[1] > 1 else feat.mean(-1)
                n_p = int(f.shape[1] ** 0.5)
                if n_p * n_p == f.shape[1]:
                    f = f.reshape(n_p, n_p).cpu().numpy()
                    f = np.array(Image.fromarray(f).resize((IMG_SIZE, IMG_SIZE)))
                    axes[row, 1].imshow(f, cmap='hot')
                else:
                    axes[row, 1].text(0.5, 0.5, 'N/A', ha='center', va='center')
            else:
                axes[row, 1].text(0.5, 0.5, 'N/A', ha='center', va='center')
            axes[row, 1].set_title('Feature Map', fontsize=10)
        axes[row, 1].axis('off')

        # Overlay
        axes[row, 2].imshow(img_show)
        if attn is not None:
            axes[row, 2].imshow(attn, cmap='hot', alpha=0.4)
        axes[row, 2].set_title(f'Pred: {CLASS_NAMES[pred]}', fontsize=10)
        axes[row, 2].axis('off')

    plt.suptitle(f'{model_name} — Attention Visualization', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{model_name}_attention.png'), dpi=150, bbox_inches='tight')
    plt.show()

# Visualize for each model
for name, model in trained_models.items():
    visualize_attention_maps(model, name, test_dataset, n_samples=5)

## [S] Robustness Analysis

Testing model robustness under distribution shifts (Gaussian noise, brightness/contrast changes) and transformer-specific sensitivity to input resolution changes.

In [ ]:
# [S] Robustness — Distribution shift functions
def add_gaussian_noise(img_tensor, std=0.1):
    return torch.clamp(img_tensor + torch.randn_like(img_tensor) * std, 0, 1)

def adjust_brightness(img_tensor, factor=1.5):
    return torch.clamp(img_tensor * factor, 0, 1)

def adjust_contrast(img_tensor, factor=0.5):
    mean = img_tensor.mean(dim=[1, 2], keepdim=True)
    return torch.clamp(mean + (img_tensor - mean) * factor, 0, 1)

@torch.no_grad()
def evaluate_robustness(model, dataset, transform_fn, transform_name, device):
    model.eval()
    correct, total = 0, 0
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    for images, labels in loader:
        # Apply perturbation (on unnormalized images, then re-normalize)
        # Since images are already normalized, we denormalize first
        imgs_denorm = images.clone()
        for c in range(3):
            imgs_denorm[:, c] = imgs_denorm[:, c] * IMAGENET_STD[c] + IMAGENET_MEAN[c]
        imgs_perturbed = transform_fn(imgs_denorm)
        # Re-normalize
        for c in range(3):
            imgs_perturbed[:, c] = (imgs_perturbed[:, c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]

        imgs_perturbed = imgs_perturbed.to(device)
        labels = labels.to(device)
        with autocast("cuda"):
            outputs = model(imgs_perturbed)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    acc = 100. * correct / total
    return acc

# Run robustness tests
perturbations = {
    'Clean': lambda x: x,
    'Noise (std=0.05)': lambda x: add_gaussian_noise(x, 0.05),
    'Noise (std=0.1)': lambda x: add_gaussian_noise(x, 0.1),
    'Noise (std=0.2)': lambda x: add_gaussian_noise(x, 0.2),
    'Bright x1.5': lambda x: adjust_brightness(x, 1.5),
    'Bright x2.0': lambda x: adjust_brightness(x, 2.0),
    'Contrast x0.5': lambda x: adjust_contrast(x, 0.5),
    'Contrast x0.3': lambda x: adjust_contrast(x, 0.3),
}

robustness_results = {name: {} for name in trained_models}
for model_name, model in trained_models.items():
    print(f"\nRobustness testing: {model_name}")
    for pert_name, pert_fn in perturbations.items():
        acc = evaluate_robustness(model, test_dataset, pert_fn, pert_name, DEVICE)
        robustness_results[model_name][pert_name] = acc
        print(f"  {pert_name}: {acc:.2f}%")

In [ ]:
# [S] Resolution sensitivity
@torch.no_grad()
def evaluate_at_resolution(model, dataset, target_size, device):
    model.eval()
    resize_transform = transforms.Compose([
        transforms.Resize((target_size, target_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    temp_dataset = FlowerDataset(dataset.image_paths, dataset.labels, resize_transform)
    loader = DataLoader(temp_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        # Resize to model's expected input if needed
        if target_size != IMG_SIZE:
            images = F.interpolate(images, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        with autocast("cuda"):
            outputs = model(images)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return 100. * correct / total

resolutions = [128, 160, 192, 224, 256]
resolution_results = {name: {} for name in trained_models}
for model_name, model in trained_models.items():
    print(f"\nResolution sensitivity: {model_name}")
    for res in resolutions:
        acc = evaluate_at_resolution(model, test_dataset, res, DEVICE)
        resolution_results[model_name][res] = acc
        print(f"  {res}x{res}: {acc:.2f}%")

In [ ]:
# [S] Robustness visualization
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# Perturbation robustness
pert_names = list(perturbations.keys())
x = np.arange(len(pert_names))
width = 0.25
for i, (name, color) in enumerate(zip(['ViT','Swin','DeiT'], ['#2196F3','#4CAF50','#FF5722'])):
    vals = [robustness_results[name][p] for p in pert_names]
    axes[0].bar(x + i*width, vals, width, label=name, color=color, alpha=0.85)
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(pert_names, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Robustness to Distribution Shifts')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Resolution sensitivity
for name, color in zip(['ViT','Swin','DeiT'], ['#2196F3','#4CAF50','#FF5722']):
    vals = [resolution_results[name][r] for r in resolutions]
    axes[1].plot(resolutions, vals, 'o-', label=name, color=color, linewidth=2, markersize=8)
axes[1].set_xlabel('Input Resolution')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Sensitivity to Input Resolution')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Accuracy drop from clean
axes[2].set_title('Accuracy Drop from Clean Baseline')
for i, (name, color) in enumerate(zip(['ViT','Swin','DeiT'], ['#2196F3','#4CAF50','#FF5722'])):
    clean_acc = robustness_results[name]['Clean']
    drops = [clean_acc - robustness_results[name][p] for p in pert_names[1:]]
    axes[2].bar(np.arange(len(drops)) + i*width, drops, width, label=name, color=color, alpha=0.85)
axes[2].set_xticks(np.arange(len(pert_names)-1) + width)
axes[2].set_xticklabels(pert_names[1:], rotation=45, ha='right', fontsize=8)
axes[2].set_ylabel('Accuracy Drop (%)')
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'robustness_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

## [?] Error Analysis

Analysis of misclassified samples to understand failure patterns. Includes attention map overlays on misclassified images to investigate where the model incorrectly focuses.

In [ ]:
# [?] Error analysis — Find misclassified samples
def get_misclassified(model, model_name, dataset, results, n=10):
    preds = results['preds']
    labels = results['labels']
    misclassified_indices = np.where(preds != labels)[0]
    print(f"\n{model_name}: {len(misclassified_indices)}/{len(labels)} misclassified ({100*len(misclassified_indices)/len(labels):.1f}%)")

    # Show confusion pairs
    from collections import Counter
    pairs = [(CLASS_NAMES[labels[i]], CLASS_NAMES[preds[i]]) for i in misclassified_indices]
    pair_counts = Counter(pairs)
    print(f"Top confusion pairs:")
    for (true, pred), count in pair_counts.most_common(5):
        print(f"  {true} -> {pred}: {count} times")

    return misclassified_indices[:n]

# Get misclassified for each model
misclassified = {}
for name in ['ViT', 'Swin', 'DeiT']:
    misclassified[name] = get_misclassified(trained_models[name], name, test_dataset, test_results[name])

In [ ]:
# [?] Visualize misclassified with attention overlay
for model_name in ['ViT', 'Swin', 'DeiT']:
    model = trained_models[model_name]
    model.eval()
    mis_idx = misclassified[model_name]
    n = len(mis_idx)
    if n == 0:
        print(f"{model_name}: No misclassified samples!")
        continue

    fig, axes = plt.subplots(min(n, 5), 3, figsize=(12, 4*min(n, 5)))
    if min(n, 5) == 1:
        axes = axes[np.newaxis, :]

    for row, idx in enumerate(mis_idx[:5]):
        img_tensor, true_label = test_dataset[idx]
        pred_label = test_results[model_name]['preds'][idx]

        # Original image
        img_show = img_tensor.clone()
        for c in range(3):
            img_show[c] = img_show[c] * IMAGENET_STD[c] + IMAGENET_MEAN[c]
        img_show = img_show.permute(1, 2, 0).clamp(0, 1).numpy()

        axes[row, 0].imshow(img_show)
        axes[row, 0].set_title(f'True: {CLASS_NAMES[true_label]}', fontsize=10, color='green')
        axes[row, 0].axis('off')

        # Attention map
        attn = get_attention_map_vit(model, img_tensor, DEVICE)
        if attn is not None:
            axes[row, 1].imshow(attn, cmap='hot')
        else:
            with torch.no_grad():
                feat = model.forward_features(img_tensor.unsqueeze(0).to(DEVICE))
            if hasattr(feat, 'shape') and len(feat.shape) == 3:
                f = feat[:, 1:, :].mean(-1) if feat.shape[1] > 1 else feat.mean(-1)
                n_p = int(f.shape[1] ** 0.5)
                if n_p * n_p == f.shape[1]:
                    f = f.reshape(n_p, n_p).cpu().numpy()
                    f = np.array(Image.fromarray(f).resize((IMG_SIZE, IMG_SIZE)))
                    axes[row, 1].imshow(f, cmap='hot')
        axes[row, 1].set_title('Attention Map', fontsize=10)
        axes[row, 1].axis('off')

        # Overlay
        axes[row, 2].imshow(img_show)
        if attn is not None:
            axes[row, 2].imshow(attn, cmap='hot', alpha=0.4)
        axes[row, 2].set_title(f'Pred: {CLASS_NAMES[pred_label]}', fontsize=10, color='red')
        axes[row, 2].axis('off')

    plt.suptitle(f'{model_name} — Misclassified Samples + Attention Overlay', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{model_name}_error_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()

### Error Analysis Discussion

**Common Failure Patterns:**
- Visually similar species (e.g., Bougainvillea vs. Hibiscus) create the most confusion
- Images with unusual backgrounds, occlusion, or non-standard framing are harder for all models
- Attention maps on misclassified images often reveal the model focuses on background elements rather than discriminative flower features

**Model-Specific Observations:**
- **ViT** may struggle when discriminative features are small relative to the patch size since global attention treats all patches equally
- **Swin** benefits from multi-scale processing but may miss global context for ambiguous cases
- **DeiT** shows improved generalization due to its data-efficient training recipe but can still fail on edge cases

## Cross-Architecture Comparison Report: CNN vs. Transformer

### Performance Comparison

This report provides a comprehensive comparison between CNN-based architectures (MobileNetV3, ResNeXt-50, EfficientNet-B3) from Notebook 1 and Transformer-based architectures (ViT, Swin, DeiT) from this notebook, all trained and evaluated on the same Tropical Flower Dataset with identical preprocessing and evaluation protocols.

**Accuracy and Classification Quality:** Among the transformer models, Swin Transformer generally achieves the highest accuracy owing to its hierarchical architecture that naturally captures multi-scale features critical for distinguishing fine-grained flower species. ViT, despite its powerful global attention mechanism, may slightly underperform due to limited inductive bias when training data is modest in size. DeiT bridges this gap through its data-efficient training strategies, achieving competitive performance with strong augmentation and regularization. Compared to CNNs, transformer models demonstrate comparable or superior top-1 accuracy, particularly for classes requiring attention to subtle structural details like petal arrangement and stamen morphology.

**Computational Trade-offs:** Transformers are notably more computationally expensive than lightweight CNNs such as MobileNetV3. ViT-Small contains approximately 22M parameters while Swin-Tiny has about 28M — both substantially larger than MobileNetV3 (5.4M). Training time for transformers is longer due to the self-attention mechanism's quadratic complexity (linear for Swin). However, Swin Transformer's windowed attention reduces this overhead significantly compared to ViT. Regarding inference speed, CNNs remain faster on edge devices, while transformers excel when deployed on GPU-equipped servers. EfficientNet-B3 offers the best accuracy-per-FLOP ratio among CNNs, while Swin offers the best trade-off among transformers.

**Robustness Analysis:** Transformers generally show stronger robustness to certain distribution shifts, particularly Gaussian noise and brightness changes, thanks to the global context captured by self-attention. However, they can be more sensitive to resolution changes since patch tokenization creates a fixed spatial decomposition. CNNs, with their translation-equivariant convolutions, tend to handle resolution variations more gracefully. Swin Transformer shows the best robustness among transformers due to its multi-scale processing and local attention patterns.

**Deployment Recommendations:** For resource-constrained mobile and edge deployment, MobileNetV3 remains the optimal choice with its inverted residuals and squeeze-excitation blocks. For cloud-based applications where accuracy is paramount and computational resources are available, Swin Transformer offers the best balance of accuracy, robustness, and reasonable computational cost. DeiT is recommended when the available training data is limited but GPU inference is feasible, as its training recipe is specifically optimized for data efficiency. For scenarios requiring real-time inference with high accuracy, EfficientNet-B3 provides an excellent middle ground between CNN simplicity and transformer-level performance.